# T4 Energy Calibration for ShadowKV++

This notebook measures actual GPU energy (NVML) on a **Google Colab T4 GPU**
for a representative subset of the ShadowKV++ controlled benchmark matrix.

**What you need:**
- The file `t4_energy_transfer.zip`
- A Colab T4 GPU runtime (Runtime → Change runtime type → T4 GPU)

**Timing:** ~40 min (quick, 20 cells) or ~72 min (full, 36 cells)

## 1. Upload the Package

Click the button below and select `t4_energy_transfer.zip` from your computer.

In [ ]:
from google.colab import files as colab_files
import zipfile, os, sys, time, glob, json
import pandas as pd

print('Please select t4_energy_transfer.zip...')
uploaded = colab_files.upload()

zip_path = None
for fname in uploaded.keys():
    if fname.endswith('.zip'):
        zip_path = fname
        break

if zip_path is None:
    raise RuntimeError('No .zip file found. Please upload t4_energy_transfer.zip')

print('Extracting %s...' % zip_path)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall('/content/')

if os.path.exists('/content/t4_energy_transfer'):
    os.chdir('/content/t4_energy_transfer')
else:
    items = [d for d in os.listdir('/content') if os.path.isdir('/content/' + d) and d != 'sample_data']
    if items:
        os.chdir('/content/' + items[0])
    else:
        raise RuntimeError('Could not find extracted directory')

print('Working directory:', os.getcwd())
print('Files:', len(os.listdir('.')))

## 2. Install Dependencies (~3-5 min)

Installs the ShadowKV++ package and Python dependencies.

In [ ]:
print('Installing dependencies...')
!pip install -U pip -q
!pip install -e . -q
!pip install pytest -q

import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
print('\nGPU:', gpu_name)
if torch.cuda.is_available():
    mem = torch.cuda.get_device_properties(0).total_mem / 1073741824
    print('Memory: %.1f GB' % mem)
    print('CUDA:', torch.version.cuda)
else:
    print('WARNING: No GPU detected. Change runtime type to T4 GPU.')

## 3. Smoke Test (CPU, ~10 seconds)

Verifies the installation is working before spending GPU time.

In [ ]:
print('Running smoke test...')
!python experiments/run_benchmark.py --backend fake --workload synthetic --n_requests 8 --include_experimental --disable_arrival_simulation --output_dir /tmp/smoke
print('\nSmoke test passed! Environment is ready.')

## 4. Run Energy Calibration Experiment

This is the main experiment. Runs the benchmark with --measure_energy
on a small matrix to get actual GPU energy readings.

In [ ]:
# --- Choose mode ---
# QUICK_MODE = True  -> 20 cells, ~40 min, 5 engines
# QUICK_MODE = False -> 36 cells, ~72 min, all 9 engines
QUICK_MODE = False

label = 'quick' if QUICK_MODE else 'full'
n_cells = 20 if QUICK_MODE else 36
est_min = 40 if QUICK_MODE else 72

print('Mode:', 'QUICK' if QUICK_MODE else 'FULL')
print('Cells:', n_cells)
print('Estimated time: ~%d min' % est_min)
r = input('Ready to start? (yes/no): ')
if r.lower() != 'yes':
    print('Cancelled. Run this cell again when ready.')
    raise SystemExit()

In [ ]:
start_time = time.time()

cmd = [sys.executable, 'experiments/run_t4_energy_calibration.py']
if QUICK_MODE:
    cmd.append('--quick')

print('Starting:', ' '.join(cmd))
print('=' * 60)

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR (last 2000 chars):')
    print(result.stderr[-2000:])

elapsed_min = (time.time() - start_time) / 60
print('Experiment finished in %.0f minutes.' % elapsed_min)

## 5. Extrapolate to Full Result Bundle

Uses the measured J/req to estimate GPU energy for all 898 controlled-result
JSONs (T4 + P100, 5 models, 10 datasets, 3 modes, 3 seeds).

In [ ]:
print('Running calibration...')
!python experiments/calibrate_energy_from_measurement.py
print('Calibration complete.')

## 6. Download Results

The results are zipped and a download link appears below.

In [ ]:
results_dir = '/content/t4_energy_transfer/results'
zip_path = '/content/t4_energy_calibration_%s.zip' % label

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(results_dir):
        for f in files:
            fp = os.path.join(root, f)
            arc = os.path.relpath(fp, results_dir)
            zf.write(fp, arc)

print('Results zipped:', zip_path)
print('Size: %.0f MB' % (os.path.getsize(zip_path) / 1048576))

In [ ]:
colab_files.download(zip_path)

## 7. Preview: Headline Energy Estimates

Aggregate energy per engine across the full controlled bundle.

In [ ]:
summary_path = '/content/t4_energy_transfer/results/t4_energy_calibration/summary_by_engine_energy.csv'
if os.path.exists(summary_path):
    df = pd.read_csv(summary_path)
    print('Headline Energy Estimates (T4 + P100 combined):')
    print(df.to_string(index=False))
else:
    print('Summary CSV not found yet.')